# Converting a keras model with HLS4ML

Check if your virtual enviroment is active!
Now let's get the dependecies ...

In [ ]:
!which python

In [ ]:
!pip install pydot graphviz sklearn

With the additional packages installed we can import the libraries needed.

In [ ]:
import hls4ml
import yaml
import numpy as np
from utils import plotting
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model

Declare the vivado path.

In [ ]:
import os
os.environ['PATH'] = '/tools/Xilinx/Vivado/2019.2/bin:' + os.environ['PATH']

To be able to compare the hls and keras model, we need to load the test data and the keras model.

In [ ]:
X_test = np.load(os.path.join('data','X_test.npy'), allow_pickle=True)
y_test = np.load(os.path.join('data','y_test.npy'), allow_pickle=True)
classes = np.load(os.path.join('data','classes.npy'), allow_pickle=True)

In [ ]:
model = load_model('jet_tagging_keras/KERAS_check_best_model.h5')

To run the prediction execute:

In [ ]:
y_keras = model.predict(X_test)

To use HLS4ML to it's full extend, the hls design can be adjusted and configurt.

These can be done by writing code or manually editing the yaml-file. 

Here the configuraton is set within the python code. 

A seperatly written file can be loaded for e.g. with
```
with open("keras-config.yml", 'r') as ymlfile:
    config = yaml.load(ymlfile, Loader=yaml.FullLoader)

```

Let's create the configuration out of the loaded model. 

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='name')
print("-----------------------------------")
plotting.print_dict(config)
print("-----------------------------------")

The network structure and the layer settings are shown above.
HLS4ML alows to adjust every layer seperatly. 

Now is the time to create the hls representation and run some profiling. A good explanation of the charts can be found [here](https://github.com/fastmachinelearning/hls4ml-tutorial/blob/master/part2_advanced_config.ipynb).
```
The first thing to try is to numerically profile your model. This method plots the distribution of the weights (and biases) as a box and whisker plot. The grey boxes show the values which can be represented with the data types used in the hls_model. Generally, you need the box to overlap completely with the whisker 'to the right' (large values) otherwise you'll get saturation & wrap-around issues. It can be okay for the box not to overlap completely 'to the left' (small values), but finding how small you can go is a matter of trial-and-error.
```

We get an overview of the network structure and the weight distribution.

In [ ]:
from hls4ml.model import profiling
hls_model = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_jt/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')

profiling.numerical(keras_model=model, hls_model=hls_model, X=X_test[:1000])

To edit entries inside the configuration, it can be used like a python dictionary.
Let's change the fixed-point size of the first layer and create the hls model again:

In [ ]:
config['LayerName']['fc1']['Precision']['weight'] = 'ap_fixed<8,2>'
hls_model = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_jt/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')
profiling.numerical(keras_model=model, hls_model=hls_model)
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

If we want to know what happens inside the model because of our changes, we need to run the trace method.

First we change the entries accordingly and create the hls model.

In [ ]:
for layer in config['LayerName'].keys():
    config['LayerName'][layer]['Trace'] = True
hls_model = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_jt/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')

Then the model needs to be compiled and we get the predictions of the individual layers.

In [ ]:
hls_model.compile()
hls4ml_pred, hls4ml_trace = hls_model.trace(np.ascontiguousarray(X_test[:1000]))

To compare it with the results of the keras model, we need to do the same here as well.

There is an import error in the profiling file which won't allow us to use get_ymodel_keras method. 

This will replace the file with a modified version.

In [ ]:
path_to_fix = os.path.join(os.getcwd(),'venv_hls4ml2','lib','python3.6','site-packages','hls4ml','model')

os.rename(os.path.join(path_to_fix,'profiling.py'),os.path.join(path_to_fix,'profiling2.py'))
import shutil
shutil.copy(os.path.join(os.getcwd(),'profiling_wo_qkeras.py'), os.path.join(path_to_fix,'profiling.py'))

In [ ]:
keras_trace = hls4ml.model.profiling.get_ymodel_keras(model,np.ascontiguousarray( X_test[:1000]))
y_hls = hls_model.predict(np.ascontiguousarray(X_test))

Now we can get the numerical representation of the predictions ...

In [ ]:
print("Keras layer 'fc1', first sample:")
print(keras_trace['fc1'][0])
print("hls4ml layer 'fc1', first sample:")
print(hls4ml_trace['fc1'][0])

Which is definitely more appealing in form of a graphical representation ...

In [ ]:
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, classes)
plt.gca().set_prop_cycle(None) # reset the colors
_ = plotting.makeRoc(y_test, y_hls, classes, linestyle='--')

from matplotlib.lines import Line2D
lines = [Line2D([0], [0], ls='-'),
         Line2D([0], [0], ls='--')]
from matplotlib.legend import Legend
leg = Legend(ax, lines, labels=['keras', 'hls4ml'],
            loc='lower right', frameon=False)
ax.add_artist(leg)

Beside the fixed-point size, an other important setting is the reuse factor. 

So let's change it for the whole model.

This won't affect the latency but the useage of MAC units and logic elements.

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='Model')
print("-----------------------------------")
print(config)
print("-----------------------------------")
# Set the ReuseFactor to 2 throughout
config['Model']['ReuseFactor'] = 2
hls_model = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_std/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')
hls_model.compile()
y_hls = hls_model.predict(np.ascontiguousarray(X_test))
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))
plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, classes)
plt.gca().set_prop_cycle(None) # reset the colors
_ = plotting.makeRoc(y_test, y_hls, classes, linestyle='--')

In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

After compilation the hls model is ready to use and 
we will run some predictions.

In [ ]:
hls_model.compile()
y_hls = hls_model.predict(X_test)

This will start vivado and run the synthesis. 
The process can take some time, to monitor the logs run the command below inside a new console:
    tail -f model_std/hls4ml_prj/vivado_hls.log

In [ ]:
hls_model.build(csim=False)

And let's have a look at the report.

In [ ]:
hls4ml.report.read_vivado_report('model_std/hls4ml_prj/')